<a href="https://colab.research.google.com/github/chantika-maharani/ProyekAkhir_5C_Estimation-of-Obesity-Levels/blob/main/Notebook/02_praproses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laporan Praproses Data untuk Pemodelan K-Means

Praproses data merupakan tahapan krusial sebelum menerapkan algoritma clustering seperti K-Means. Karena K-Means menggunakan perhitungan jarak geometris (*Euclidean Distance*), performa model sangat dipengaruhi oleh perbedaan skala fitur dan keberadaan data kategorikal. Berikut adalah alur praproses data terstruktur yang dirancang agar hasil pemodelan menjadi valid.

### Langkah 1: Impor Library dan Memisahkan Fitur Utama
Kami mengimpor modul `KMeans` dari `sklearn.cluster` untuk pemodelan clustering. Pada analisis tanpa pengawasan, variabel target asli (`NObeyesdad`) dikeluarkan dari proses klasterisasi, namun tetap disimpan secara terpisah sebagai referensi pembanding di akhir.

In [10]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("/content/data_bersih (1).csv")

print("Ukuran dataset:", df.shape)
display(df.head())

Ukuran dataset: (2087, 17)


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


### Langkah 2: Identifikasi Tipe Atribut (Numerik vs Kategorikal)
Atribut numerik dan kategorikal memerlukan penanganan yang berbeda. Kami mengelompokkan jenis kolom ini secara otomatis berdasarkan tipe datanya.

In [11]:
fitur = ["FAVC", "FCVC", "CAEC", "FAF", "TUE"]

X = df[fitur].copy()

print("Fitur yang digunakan:")
print(X.columns.tolist())

display(X.head())
print("Missing value:")
print(X.isnull().sum())

print("\nData duplikat:", X.duplicated().sum())

print("\nTipe data:")
print(X.dtypes)

Fitur yang digunakan:
['FAVC', 'FCVC', 'CAEC', 'FAF', 'TUE']


,FAVC,FCVC,CAEC,FAF,TUE
0,no,2.0,Sometimes,0.0,1.0
1,no,3.0,Sometimes,3.0,0.0
2,no,2.0,Sometimes,2.0,1.0
3,no,3.0,Sometimes,2.0,0.0
4,no,2.0,Sometimes,0.0,0.0


Missing value:
FAVC    0
FCVC    0
CAEC    0
FAF     0
TUE     0
dtype: int64

Data duplikat: 416

Tipe data:
FAVC     object
FCVC    float64
CAEC     object
FAF     float64
TUE     float64
dtype: object


### Langkah 2: Konstruksi Pipeline Praproses Data
Karena K-Means mengandalkan jarak geometris, semua fitur harus berupa numerik berskala seragam:
1. **Numerik**: Diimputasi dengan *median* dan dinormalisasi menggunakan `StandardScaler`.
2. **Kategorikal**: Diimputasi dengan *modus* lalu dikonversi menjadi biner via `OneHotEncoder`.

In [12]:
fitur_numerik = ["FCVC", "FAF", "TUE"]

fitur_kategorikal = ["FAVC", "CAEC"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numerik", StandardScaler(), fitur_numerik),
        ("kategori", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), fitur_kategorikal)
    ]
)

X_scaled = preprocessor.fit_transform(X)

nama_fitur = preprocessor.get_feature_names_out()

X_preprocessed = pd.DataFrame(
    X_scaled,
    columns=nama_fitur
)

display(X_preprocessed.head())

print("Ukuran data:", X_preprocessed.shape)

,numerik__FCVC,numerik__FAF,numerik__TUE,kategori__FAVC_no,kategori__FAVC_yes,kategori__CAEC_Always,kategori__CAEC_Frequently,kategori__CAEC_Sometimes,kategori__CAEC_no
0,-0.788364,-1.186977,0.554211,1.0,0.0,0.0,0.0,1.0,0.0
1,1.082164,2.328908,-1.090505,1.0,0.0,0.0,0.0,1.0,0.0
2,-0.788364,1.156947,0.554211,1.0,0.0,0.0,0.0,1.0,0.0
3,1.082164,1.156947,-1.090505,1.0,0.0,0.0,0.0,1.0,0.0
4,-0.788364,-1.186977,-1.090505,1.0,0.0,0.0,0.0,1.0,0.0


Ukuran data: (2087, 9)


In [14]:
# Menyimpan data hasil preprocessing
X_preprocessed.to_csv(
    "data_preprocessed.csv",
    index=False
)


print("Preprocessing berhasil disimpan")

Preprocessing berhasil disimpan


# Ringkasan Hasil Praproses Data

Berikut adalah ringkasan teknis dari tahapan praproses data pada dataset `/content/data_bersih (1).csv`:

1. **Pemisahan Fitur Utama**:
   * Variabel target asli (`NObeyesdad`) dikeluarkan dari dataset fitur utama agar algoritma clustering dapat berfokus memetakan data berdasarkan kemiripan karakteristik tanpa adanya intervensi dari label klasifikasi.

2. **Konstruksi Pipeline Praproses (Standardisasi Ketat)**:
   * **Fitur Numerik**: Diimputasi menggunakan metode *median* dan dinormalisasi menggunakan `StandardScaler` agar memiliki skala rata-rata 0 dan variansi 1. Ini sangat penting agar fitur dengan skala besar tidak mendominasi perhitungan jarak.
   * **Fitur Kategorikal**: Diimputasi dengan nilai *modus* (kategori paling sering muncul) dan ditransformasikan secara aman menjadi representasi biner menggunakan `OneHotEncoder`.

3. **Dimensi Akhir Fitur**:
   * Hasil transformasi praproses menghasilkan matriks data akhir berdimensi **(2087, 17)** yang siap digunakan untuk proses analisis klaster di file atau tahap terpisah.